$\Huge\text{Character-Level Text Prediction}$

$\huge\text{Artificial Neural Networks Project}$

$\large\text{Jorge Rodríguez Toledo}$


# PACKAGES AND DATA LOADING

Here we import all the packages needed for the analysis as well as the data that will be used!

In [1]:
# Timing, randon numbers and storing data
import time, random, h5py 

# python essentials
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load dataset 
filepath = "tinyshakespeare.txt"
with open(filepath, "r", encoding="utf-8") as file:
    text = file.read()
print(f"Total characters: {len(text):,}")

2026-05-12 11:57:57.623515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778579877.635314   12009 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778579877.639530   12009 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778579877.648831   12009 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778579877.648846   12009 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778579877.648848   12009 computation_placer.cc:177] computation placer alr

Total characters: 1,115,394


# PREPROCESSING 

In this step, we fix the seed, define the training windows for the models and perform a 90-10 train-validation split

In [2]:
# Ensures reproducibility
seed = 5721
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Extract unique characters and sort them in ascending order
characters = sorted(list(set(text)))   
vocab_size = len(characters)
print(f"Unique characters: {characters}")
print(f"Vocabulary size: {vocab_size}")

# TRANSLATION LAYER
# ML models work best with numbers so we encode the characters as integers
# To generate text, we then have to translate the integers back to characters
char_to_idx = {ch: i for i, ch in enumerate(characters)} # characters to integers 
idx_to_char = {i: ch for ch, i in char_to_idx.items()} # integers to characters

# Encode the full text 
encoded = np.array([char_to_idx[c] for c in text], dtype=np.int32)


# TRAINING WINDOWS 
# We train the model to predict the next character given a sequence of previous characters
# We choose a fixed length for the input sequence and slide it across the full text
# HYPERPARAMETERS
sequence_length = 100 # context window length
step = sequence_length//3  # stride, to ensure overlapping windows
batchist_s = 128 # number of independent sequences processed at once

# TRAIN-VALIDATION SPLIT (90/10)
n_train = int(len(encoded)*0.9)
encoded_train = encoded[:n_train]
encoded_val = encoded[n_train:]

# INPUT-TARGET SPLIT
# Each input sequence is paired with a target sequence
# This is the same as the input but shifted one character to the right
def input_target(sequence):
    return sequence[:-1], sequence[1:]

# Creates the batched dataset
def make_dataset(encoded_array, shuffle=True):
    individual_characters = tf.data.Dataset.from_tensor_slices(encoded_array) # slices the data into individual characters
    windows = individual_characters.window(sequence_length+1, shift=step, drop_remainder=True) # defines the windows dataset
    sequences = windows.flat_map(lambda w: w.batch(sequence_length+1)) # flattens the windows into sequences
    input_target_pairs = sequences.map(input_target) # creates the input-target pairs
    if shuffle:
        input_target_pairs = input_target_pairs.shuffle(buffer_size=10_000, seed=seed)
    return input_target_pairs.batch(batchist_s, drop_remainder=True).prefetch(tf.data.AUTOTUNE) # batches the data and prefetches it for performance

# Make the train-val datassets
train_dataset = make_dataset(encoded_train, shuffle=True) # Shuffling ensures the model doesn't memorize the order (generalization)
val_data = make_dataset(encoded_val, shuffle=False)
n_train_batches = sum(1 for _ in train_dataset)
n_val_batches = sum(1 for _ in val_data)
print("")
print(f"Train batches: {n_train_batches}")
print(f"Validation batches: {n_val_batches}")

Unique characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary size: 65


I0000 00:00:1778579882.887404   12009 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4151 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
2026-05-12 11:58:07.021843: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Train batches: 237
Validation batches: 26


2026-05-12 11:58:07.359002: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# MODELS

Defines a function that builds all 3 models using the same hyperparameters

All models have the same architecture:
1. Input layer
2. Embedding layer
3. 2 hidden + dropout layers
4. an optional self-attention layer
5. output layer

In [3]:
# UNIVERSAL HYPERPARAMETERS
d=64 # embedding dimension 
h=128 # hidden layer dimension
n=2 # number of layers

# MODELS
recurrent_cell = {
    'rnn': layers.SimpleRNN, 
    'gru': layers.GRU, 
    'lstm': layers.LSTM
}

# Builds the recurrent model
def build_model(vocab_size, model_type='lstm', embed_dim=64, hidden_size=128, n_layers=2, use_attention=True):
    # STEP 1: INPUT LAYER
    inputs = keras.Input(shape=(None,), dtype=tf.int32, name="char_ids") # accepts any length sequence, and any batch size
    
    # STEP 2: EMBEDDING LAYER
    # It converts integers into dense vectors (e.g., 0 -> [0.1, 0.2, ..., 0.3])
    # this allows the model to learn a more meaningful representation of characters
    x = layers.Embedding(vocab_size, embed_dim, name="embedding")(inputs)

    # STEP 3: HIDDEN+DROPOUT LAYERS
    # Stacks n recurrent layers followed by droppout layers for regularizstion
    RecurrentLayer = recurrent_cell[model_type.lower()] # selects recurrent layer
    for i in range(n_layers):
        x = RecurrentLayer(hidden_size, return_sequences=True, recurrent_initializer="glorot_uniform", name=f"{model_type}_{i+1}")(x) 
        x = layers.Dropout(0.2, name=f"dropout_{i+1}")(x) # model forgets some data to prevent overfitting

    # STEP 4: SELF-ATTENTION LAYER
    # It lets each position in the sequence attend to all other positions
    # so it allows the model to capture long-range dependencies
    if use_attention:
        x = layers.Attention(use_scale=True, name="attention")([x, x]) # Scales scores to prevent vanishinf gradients

    # STEP 5: OUTPUT LAYER
    # Outputs a vector of logits (vocab_size=65) for each element in the sequence (L=100)
    # each logit corresponds to a character in the vocabulary
    # (highest value -> most likely character according to the model's prediction)
    outputs = layers.Dense(vocab_size, name="logits")(x)
    # NOTE: softmax is only applied during text generation, not in training

    return keras.Model(inputs, outputs, name=f"Char{model_type.upper()}") 



# MODELS SUMMARY
for model_cell in ['rnn', 'gru', 'lstm']:
    for attention_model in [True, False]:
        build_model(vocab_size, model_type=model_cell, use_attention=attention_model).summary()

Model: "CharRNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ char_ids            │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │      4,160 │ char_ids[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_1 (SimpleRNN)   │ (None, None, 128) │     24,704 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, None, 128) │          0 │ rnn_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_2 (SimpleRNN)   │ (None, None, 128) │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, None, 128) │          0 │ rnn_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 128) │          1 │ dropout_2[0][0],  │
│ (Attention)         │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, None, 65)  │      8,385 │ attention[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 70,146 (274.01 KB)

 Trainable params: 70,146 (274.01 KB)

 Non-trainable params: 0 (0.00 B)

...

Model: "CharLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_ids (InputLayer)           │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 64)       │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, None, 128)      │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, None, 65)       │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 242,945 (949.00 KB)

 Trainable params: 242,945 (949.00 KB)

 Non-trainable params: 0 (0.00 B)

# TRAINING

We train each model monitoring the sparse categorical cross-entropy as well as the time it takes for the full training process.

We have designed a custom early stopping rule. It monitors val_loss and the learning rate is halved after 3 non-improving epochs down to 1e-5.

In [ ]:
# HYPERPARAMETERS
max_epochs = 150 # just in case, but early stopping should trigger first
learning_rate = 1e-3

# Dictionaries to store results
all_histories = {}
all_models = {} 

for model_cell in ['rnn', 'gru', 'lstm']:
    for attention_model in [True, False]:

        variant = "attention" if attention_model else "baseline"
        model_variant = f"{model_cell}_{variant}"
        checkpoint_path = f"{model_variant}.weights.h5"      
        history_path = f"history_{model_variant}.h5"

        print(f"Training {model_cell.upper()} – {variant.upper()}")

        # Builds the model
        current_model = build_model(vocab_size, model_type=model_cell, embed_dim=d, hidden_size=h, n_layers=n, use_attention=attention_model)
        current_model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate), # uses the Adam optimizer
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), # monitors cross-entropy loss
            metrics=["sparse_categorical_accuracy"] # tracks accuracy of predictions
        )

        # CALLBACKS
        # Save best validation-loss checkpoint
        checkpoint_callback = keras.callbacks.ModelCheckpoint(checkpoint_path, save_weights_only=True, monitor="val_loss", save_best_only=True, verbose=0)

        # Stop when val_loss stops improving for 8 consecutive epochs
        # restores the best weights
        earlyStopping_callback = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=0)

        # Decrease learning rate when approaching a plateau in validation loss
        learningRate_callback = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=0)

        # TRAINING
        t0 = time.time() # begin clock
        current_history = current_model.fit(train_dataset, validation_data=val_data, epochs=max_epochs, callbacks=[checkpoint_callback, earlyStopping_callback, learningRate_callback], verbose=0)
        delta_t = time.time()-t0 # stop clock

        # Store the history and training time for the current model
        history_dict = {
            'history': current_history.history,
            'epoch': current_history.epoch,
            'time': delta_t
        }
        
        # Save history as an external file
        with h5py.File(history_path, "w") as file:
            grp = file.create_group("metrics")
            for key, values in history_dict['history'].items():
                grp.create_dataset(key, data=values)
            file.create_dataset("epoch", data=history_dict['epoch'])
            file.attrs["time_seconds"] = history_dict['time']
            file.attrs["total_epochs"] = len(history_dict['epoch'])
            file.attrs["use_attention"] = int(attention_model)
            file.attrs["model_cell"] = model_cell

        # Stores results 
        all_histories[model_variant] = history_dict
        all_models[model_variant] = current_model

        # Prints results for the current model 
        metrics = history_dict['history']
        print(f"  done — {len(metrics['loss'])} epochs | "
              f"best val_loss={min(metrics['val_loss']):.4f} | "
              f"best val_acc={max(metrics['val_sparse_categorical_accuracy']):.4f} | "
              f"time={delta_t/60:.1f} min")



# Results Summary Table
print("")
print(f"{'Run':<22} {'Ep':>4} {'Val Loss':>10} {'Val Acc':>10} {'Time (min)':>12}")
print("-"*62)
for key in ['rnn_attention', 'rnn_baseline',
            'gru_attention', 'gru_baseline',
            'lstm_attention', 'lstm_baseline']:
    hist_dict = all_histories[key]
    metrics = hist_dict['history']
    print(f"{key:<22} "
          f"{len(metrics['loss']):>4} "
          f"{min(metrics['val_loss']):>10.4f} "
          f"{max(metrics['val_sparse_categorical_accuracy']):>10.4f} "
          f"{hist_dict['time']/60:>12.1f}")

Training RNN – ATTENTION


2026-05-10 00:16:08.788519: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-10 00:16:15.253007: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_199', 844 bytes spill stores, 844 bytes spill loads

2026-05-10 00:16:18.896938: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 11990826374382767229
/home/jtoledo/miniforge3/envs/machine-learning/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2026-05-10 00:16:

  done — 74 epochs | best val_loss=1.7702 | best val_acc=0.4827 | time=6.3 min
Training RNN – BASELINE


2026-05-10 00:22:28.295990: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-10 00:22:33.490894: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_188', 844 bytes spill stores, 844 bytes spill loads

2026-05-10 00:22:37.129776: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 11990826374382767229
2026-05-10 00:22:37.129793: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 13009553375254029758
2026-05-10 00:22:37.432758: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-10 00:22:39.733969: I exte

  done — 81 epochs | best val_loss=1.6589 | best val_acc=0.5098 | time=7.0 min
Training GRU – ATTENTION


...

2026-05-10 00:37:05.182590: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 11875668946957702551
2026-05-10 00:37:05.182619: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 15980848443413383334
2026-05-10 00:37:05.182649: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 17808269207473178092
2026-05-10 00:37:05.798751: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 5105851258725985835
2026-05-10 00:37:05.798768: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 564355425119228510
2026-05-10 00:37:05.798772: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 16155333221424706436
2026-05-10 00:37:05.798775: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv 

  done — 63 epochs | best val_loss=1.5419 | best val_acc=0.5235 | time=5.2 min
Training LSTM – BASELINE


2026-05-10 00:42:16.915501: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 4765130588793609922
2026-05-10 00:42:16.915515: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 607910141857417118
2026-05-10 00:42:16.915518: I tensorflow/core/framework/local_rendezvous.cc:430] Local rendezvous send item cancelled. Key hash: 2721399901449241189
2026-05-10 00:42:16.915526: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 17969590845843223373
2026-05-10 00:42:16.915528: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 1927279477947176777
2026-05-10 00:42:17.497705: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 3865954911439481443
2026-05-10 00:42:17.497719: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv ite

  done — 70 epochs | best val_loss=1.5283 | best val_acc=0.5267 | time=5.6 min

Run                      Ep   Val Loss    Val Acc   Time (min)
--------------------------------------------------------------
rnn_attention            74     1.7702     0.4827          6.3
rnn_baseline             81     1.6589     0.5098          7.0
gru_attention            41     1.5626     0.5130          3.5
gru_baseline             49     1.5340     0.5255          4.1
lstm_attention           63     1.5419     0.5235          5.2
lstm_baseline            70     1.5283     0.5267          5.6


2026-05-10 00:47:45.830039: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 3865954911439481443
2026-05-10 00:47:45.830054: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8339129575710567387
2026-05-10 00:47:45.830058: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 11305068451420694177
2026-05-10 00:47:45.830067: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 11582217329793853476


# NAIVE BASELINES

We calculate the predictions for two reference point models to compare with our trained models

In [ ]:
# UNIFORM RANDOM MODEL
# It predicts every character with equal probability (p=1/65)

# Loss (cross-entropy) for this model is:
uniform_loss = np.log(vocab_size) # -log(1/vocab_size) 
# Accuracy will be the probability of randomly guessing the correct character
uniform_accuracy = 1.0/vocab_size 

print("UNIFORM RANDOM MODEL")
print(f" Loss = {uniform_loss:.4f} nats")
print(f" Accuracy = {uniform_accuracy*100:.2f}%")


# UNIGRAM BASELINE 
# It predicts the character according to the frequency distribution of each 
# character present in the training data.

# We first determine the character frequencies
character_counts = np.bincount(encoded_train, minlength=vocab_size).astype(float) 
total_counts = character_counts.sum() 
character_frequencies = character_counts/total_counts 

# Prints top 10 most frequent characters
print("")
print("Character frequency table (top 10):")
top10 = np.argsort(character_frequencies)[::-1][:10]
for rank, idx in enumerate(top10, 1):
    print(f" {rank:2d}. {repr(idx_to_char[idx]):6s} {character_frequencies[idx]*100:.2f}%")

# Loss (cross-entropy) is the Shannon entropy
#  H(p)= E[-log p]
unigram_loss     = float(-np.sum(character_frequencies*np.log(character_frequencies+1e-12)))  # H(p)=-Sum(p*log(p))
# Accuracy is a bit harder
# Lets say we predict the most-frequent character so our accuracy will be
#  Accuracy = freq(most common char)
most_common_idx = int(np.argmax(character_frequencies))
unigram_accuracy = float(character_frequencies[most_common_idx])

print("")
print("UNIGRAM BASELINE")
print(f" Loss = {unigram_loss:.4f} nats")
print(f" Accuracy = {unigram_accuracy*100:.2f}%")


UNIFORM RANDOM MODEL
 Loss = 4.1744 nats
 Accuracy = 1.54%

Character frequency table (top 10):
  1. ' '    15.27%
  2. 'e'    8.52%
  3. 't'    6.02%
  4. 'o'    5.93%
  5. 'a'    4.95%
  6. 'h'    4.62%
  7. 's'    4.46%
  8. 'r'    4.41%
  9. 'n'    4.36%
 10. 'i'    4.08%

UNIGRAM BASELINE
 Loss = 3.3091 nats
 Accuracy = 15.27%


# FIGURES 

We plot 3 figures: model comparison, best model train vs validation and attention comparisoin

In [12]:
# Styling figures
plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'cm', 'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11, 'axes.titleweight': 'normal', 'legend.fontsize': 9.5, 'legend.framealpha': 0.9, 'legend.edgecolor': '0.75', 'xtick.labelsize': 10, 'ytick.labelsize': 10, 'xtick.direction': 'in', 'ytick.direction': 'in', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.75, 'grid.linestyle': ':', 'grid.alpha': 0.45, 'lines.linewidth': 1.75, 'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.04})
cells = ['rnn', 'gru', 'lstm']
labels = {'rnn': 'Simple RNN', 'gru': 'GRU',  'lstm': 'LSTM'}
cell_colors = {'rnn': '#D62728', 'gru': '#FF7F0E', 'lstm': '#1F77B4'}
linestyle = {'rnn': '--', 'gru': '-.', 'lstm': '-'}
variants = ['attention', 'baseline']
variant_colors = {'baseline': '#AEC7E8', 'attention': '#1F77B4'}

# Loads historyu from external files
all_histories = {}
for cell in cells:
    for variant in variants:
        key = f"{cell}_{variant}"
        with h5py.File(f"history_{key}.h5", "r") as f:
            all_histories[key] = {
                'history': {k: f["metrics"][k][:].tolist() for k in f["metrics"]},
                'epoch': f["epoch"][:].tolist(),
                'time': float(f.attrs["time_seconds"]),
                'total_epochs': int(f.attrs["total_epochs"]),
                'use_attention': bool(f.attrs["use_attention"]),
            }


# FIGURE 1: VALIDATION CURVES
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

variant_ls = {'attention': '-', 'baseline': '--'}
variant_alpha = {'attention': 1.0, 'baseline': 0.65}

for cell in cells:
    for variant in variants:
        key = f"{cell}_{variant}"
        hist_ = all_histories[key]['history']
        eps = range(1, len(hist_['val_loss']) + 1)
        label = f"{labels[cell]} ({'attention' if variant == 'attention' else 'baseline'})"
        axes[0].plot(eps, hist_['val_loss'], color=cell_colors[cell], linestyle=variant_ls[variant], alpha=variant_alpha[variant], label=label)
        axes[1].plot(eps, hist_['val_sparse_categorical_accuracy'], color=cell_colors[cell], linestyle=variant_ls[variant], alpha=variant_alpha[variant], label=label)

for ax, title, ylabel in zip(axes, ['Validation loss', 'Validation accuracy'], ['Loss', 'Accuracy']):
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig('validation_curves.pdf')
plt.close()
print('Saved validation_curves.pdf')




# FIGURE 2: BEST MODEL 
best_key = min([k for k in all_histories if k.endswith('_baseline')], key=lambda k: min(all_histories[k]['history']['val_loss']))
hist_best = all_histories[best_key]['history']
eps2 = range(1, len(hist_best['val_loss'])+1)
label = best_key.replace('_baseline', '').upper()+' (baseline)'

fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
axes2[0].plot(eps2, hist_best['loss'], color="#1F77B4", linestyle='-', label='Train')
axes2[0].plot(eps2, hist_best['val_loss'], color='#D62728', linestyle='--', label='Validation')
axes2[0].set_title(f'{label} - loss')
axes2[0].set_xlabel('Epoch')
axes2[0].set_ylabel('Loss')
axes2[0].legend()
axes2[0].grid(True)

axes2[1].plot(eps2, hist_best['sparse_categorical_accuracy'], color='#2CA02C', linestyle='-',  label='Train')
axes2[1].plot(eps2, hist_best['val_sparse_categorical_accuracy'], color='#D62728', linestyle='--', label='Validation')
axes2[1].set_title(f'{label} - accuracy')
axes2[1].set_xlabel('Epoch')
axes2[1].set_ylabel('Accuracy')
axes2[1].legend()
axes2[1].grid(True)

plt.tight_layout()
plt.savefig('best_training_curves.pdf')
plt.close()
print('Saved best_training_curves.pdf')





# FIGURE 3: ATTENTION-BASELINE COMPARISON
val_loss_data = {cell: {v: min(all_histories[f"{cell}_{v}"]['history']['val_loss']) for v in variants} for cell in cells}
val_acc_data = {cell: {v: max(all_histories[f"{cell}_{v}"]['history']['val_sparse_categorical_accuracy']) for v in variants} for cell in cells}

x = np.arange(len(cells))
width = 0.35
labels = [labels[c] for c in cells]

fig3, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4.5))

# LOSS BARS 
bars_attention = ax_loss.bar(x-width/2, [val_loss_data[c]['attention'] for c in cells], width, label='Attention', color=variant_colors['attention'], edgecolor='white')
bars_baseline = ax_loss.bar(x+width/2, [val_loss_data[c]['baseline'] for c in cells], width, label='Baseline', color=variant_colors['baseline'], edgecolor='white', hatch='//')
# Values on top of bars
for bar in bars_attention:
    ax_loss.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8.5)
for bar in bars_baseline:
    ax_loss.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8.5)

ax_loss.set_xticks(x)
ax_loss.set_xticklabels(labels)
ax_loss.set_ylabel('Loss')
ax_loss.set_title('Validation Loss')
ax_loss.legend()
ax_loss.set_ylim(
    min(val_loss_data[c][v] for c in cells for v in variants)-0.05,
    max(val_loss_data[c][v] for c in cells for v in variants)+0.08,
)
ax_loss.grid(axis='y')

# ACCURACY BARS
bars_attention2 = ax_acc.bar(x-width/2, [val_acc_data[c]['attention'] for c in cells], width, label='Attention', color=variant_colors['attention'], edgecolor='white')
bars_baseline2 = ax_acc.bar(x+width/2, [val_acc_data[c]['baseline'] for c in cells], width, label='Baseline', color=variant_colors['baseline'], edgecolor='white', hatch='//')
# Values on top of bars
for bar in bars_attention2:
    ax_acc.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8.5)
for bar in bars_baseline2:
    ax_acc.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8.5)

ax_acc.set_xticks(x)
ax_acc.set_xticklabels(labels)
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Validation Accuracy')
ax_acc.legend()
ax_acc.set_ylim(
    min(val_acc_data[c][v] for c in cells for v in variants) - 0.02,
    max(val_acc_data[c][v] for c in cells for v in variants) + 0.03,
)
ax_acc.grid(axis='y')

plt.tight_layout()
plt.savefig('attention_comparison.pdf')
plt.close()
print('Saved attention_comparison.pdf')

Saved validation_curves.pdf
Saved best_training_curves.pdf
Saved attention_comparison.pdf


# TEXT GENERATION

We use an autoregressive method that works in the following way:
1. Takes the seed and lets the model predict the next character
2. Adds this new character to the input 
3. Predicts the next character
4. Cycle repeats until number of characters is reached

In [4]:
# Rebuidls the models from the weight checkpoint files
all_models = {}

for cell in ['rnn', 'gru', 'lstm']:
    for attention_model in [True, False]:
        variant = "attention" if attention_model else "baseline"
        key = f"{cell}_{variant}"
        # builds the model with the same layers, but not the weights
        model = build_model(vocab_size, model_type=cell, embed_dim=d, hidden_size=h, n_layers=n, use_attention=attention_model)
        # we need to feed the model with some data to load the weights
        # this creates the models weight matrices, that allows to load the weights
        _ = model(tf.zeros((1, 1), dtype=tf.int32), training=False)
        model.load_weights(f"{key}.weights.h5") # loads the weights
        all_models[key] = model # stores the model

# AUTOREGRESSIVE TEXT GENERATION
def generate_text(model, seed_text, n_chars=200, temperature=1.0):   
    input_ids = [char_to_idx.get(c, 0) for c in seed_text] # Converts the seed to integers
    generated = seed_text # stores the total generated text, including the seed

    for _ in range(n_chars):
        # Feed the full current context
        input = tf.constant([input_ids])
        # Take logits for the last charctet in the sequence 
        logits = model(input, training=False)[0, -1].numpy()
        logits = logits/temperature # Temperature scaling 
        logits -= logits.max() # prevents overflow 
        probabilities = np.exp(logits)/np.exp(logits).sum() # softmax 

        # Predicts next character 
        next_id = np.random.choice(len(characters), p=probabilities)
        generated += idx_to_char[next_id] # saves generated character
        input_ids.append(next_id) # updates input
    return generated



# GENERATED SAMPLES FOR LSTM 
best_model = all_models['lstm_baseline']
print('LSTM (BASELINE) SAMPLES')
for seed, n, T, label in [
    ('To be',    50,  0.8, "Seed='To be'    | T=0.8 | ~50  chars"),
    ('ROMEO:',   200, 1.0, "Seed='ROMEO:'   | T=1.0 | ~200 chars"),
    ('The king', 200, 0.5, "Seed='The king' | T=0.5 | ~200 chars"),
    ('zzz',      100, 2.0, "FAILURE CASE Seed='zzz' | T=2.0 | ~100 chars"),
]:
    print("")
    print(f"[{label}]")
    print(generate_text(best_model, seed, n_chars=n, temperature=T))




# COMPARISON SAMPLES FOR ALL MODELS
print("")
print("MODEL COMPARISON  |  seed: \"To be\"  |  T=0.8  |  ~100 chars")
for cell in ['rnn', 'gru', 'lstm']:
    key = f"{cell}_baseline"
    text = generate_text(all_models[key], 'To be', n_chars=100, temperature=0.8)
    print("")
    print(f"[{cell.upper()} + No Attention]")
    print(f"{text}")

/home/jtoledo/miniforge3/envs/machine-learning/lib/python3.12/site-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
I0000 00:00:1778579895.620885   12077 cuda_dnn.cc:529] Loaded cuDNN version 91002


LSTM (BASELINE) SAMPLES

[Seed='To be'    | T=0.8 | ~50  chars]
To be my disseath, and confess me to a tide
Of it not u

[Seed='ROMEO:'   | T=1.0 | ~200 chars]
ROMEO:
We not, to the part, there; for thou langue
And sufferate guck his receices too resign.

RAKEN:
I say you are anobedity.

YORK:
Go are you; And he's tears, and of her,
Go well much brifll in in the m

[Seed='The king' | T=0.5 | ~200 chars]
The king have the death is great more strength that he will be honour.

ROMEO:
So stay to said the day of the saves,
And not learn a state and the contrary.

LUCIO:
There is the rest with the strength of our 

[FAILURE CASE Seed='zzz' | T=2.0 | ~100 chars]
zzzay;.--Aater yea, &hay
o' Stiffer Iscuess tozel year. Seak!
Mefissihe! his pains ban Arskun'd,
Ismend

MODEL COMPARISON  |  seed: "To be"  |  T=0.8  |  ~100 chars

[RNN + No Attention]
To be so many much liege?

BENVOLIO:
But who and the ramrer benore we shall,
Were gentles of never gast; 

[GRU + No Attention]
To be my lord,
Both a

# TEMPERATURE EFFECT ON TEXT GENERATION

We explore the effect of temperature on text generation

In [ ]:
def explore_temperatures(model, seed='To be', temperatures=(0.3, 0.6, 0.8, 1.0, 1.2, 1.5), n_chars=120):
    print(f"Temperature Exploration — LSTM (BASELINE) |  seed: {seed!r}")
    for T in temperatures:
        text = generate_text(model, seed, n_chars=n_chars, temperature=T)
        print("")
        print(f"[T={T}]")
        print(f"{text}")

explore_temperatures(best_model, seed='HAMLET:', n_chars=120)

Temperature Exploration — LSTM (BASELINE) |  seed: 'HAMLET:'

[T=0.3]
HAMLET:
So will you have the brother was more many a life
And the man that a sense the fair death,
And see the dishonour'd more

[T=0.6]
HAMLET:
No, remember me the subject of my husband.

Nurse:
The day to some the man brother of the rest
The sight. Come in the s

[T=0.8]
HAMLET:
A tell him to be my crowns of deadly to be with me
Since to a shouch as the crown are a satisportal.
Ay, in his friends

[T=1.0]
HAMLET:
He sweet his disporans with the wind cry.
Why not I proace so worn; and thou go we can.

LADY GREY:
Sir, I was penumies

[T=1.2]
HAMLET:
There in her our canst, my-chang.

ANGELO:
Nay, it is more foul men be fearful, sweet,
I were I have earss recoons, hee

[T=1.5]
HAMLET:
Tell him and slewhey; hen? 'mong;
Proud fring down: when His mamish, and I faintuart.

CLARENCE:
Brion?

NLANTES:
Nide.
